In [ ]:
# Install pymoo (if needed)
!pip install -U pymoo

import numpy as np
import matplotlib.pyplot as plt
from pymoo.problems import get_problem
from pymoo.indicators.igd import IGD
from pymoo.indicators.hv import Hypervolume

# Use a dark theme for all plots with attractive colors
plt.style.use('dark_background')


###############################################################################
# Evaluator for WFG problems
###############################################################################
class WFGEvaluator:
    def __init__(self, problem_name, n_var, n_obj):
        """
        Evaluator for WFG problems.

        :param problem_name: Name of the problem (e.g., "wfg1")
        :param n_var: Number of decision variables
        :param n_obj: Number of objectives
        """
        self.n_var = n_var
        self.n_obj = n_obj
        self.problem_name = problem_name
        self.problem = get_problem(problem_name, n_var=n_var, n_obj=n_obj)

    def evaluate(self, x):
        """Evaluate a solution 'x' and return its objective values."""
        return self.problem.evaluate(x)

    def get_true_pareto(self):
        """
        Retrieve the true Pareto front (if available).
        Raises an exception if the Pareto front is not predefined.
        """
        if hasattr(self.problem, "pareto_front"):
            return self.problem.pareto_front()
        else:
            raise NotImplementedError("This WFG problem does not have a predefined Pareto front.")

    def get_bounds(self):
        """Return the lower and upper bounds for the decision variables."""
        return self.problem.xl, self.problem.xu

    def ideal_point(self):
        """Return the ideal point of the problem."""
        return self.problem.ideal_point()

    def nadir_point(self):
        """Return the nadir point of the problem."""
        return self.problem.nadir_point()


###############################################################################
# Fast-NSGA-II with Adaptive Crossover Operators and Diversity Injection
###############################################################################
class FastNSGAII:
    def __init__(self, population_size, n_var, n_obj, max_generations,
                 crossover_rate, mutation_rate, evaluator):
        """
        Initialize Fast-NSGA-II.

        :param population_size: Population size
        :param n_var: Number of decision variables
        :param n_obj: Number of objectives
        :param max_generations: Maximum number of generations
        :param crossover_rate: Probability of performing crossover
        :param mutation_rate: Mutation probability per variable
        :param evaluator: An instance of WFGEvaluator for objective evaluations
        """
        self.population_size = population_size
        self.n_var = n_var
        self.n_obj = n_obj
        self.max_generations = max_generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.evaluator = evaluator

        # Adaptive crossover operators: SBX, One-Point, Two-Point, Uniform.
        self.crossover_ops = ["sbx", "one_point", "two_point", "uniform"]
        self.crossover_probs = {op: 1.0 / len(self.crossover_ops) for op in self.crossover_ops}
        self.learning_rate = 0.1  # learning rate to update operator probabilities

        # Initialize the population and evaluate the initial objectives.
        self.population = self.initialize_population()
        self.objectives = self.evaluate_population(self.population)

    def initialize_population(self):
        """Initialize the population uniformly within the decision bounds."""
        xl, xu = self.evaluator.get_bounds()
        pop = np.random.uniform(xl, xu, (self.population_size, self.n_var))
        return np.clip(pop, xl, xu)

    def evaluate_population(self, population):
        """Evaluate all individuals in the population."""
        return np.array([self.evaluator.evaluate(ind) for ind in population])

    # --------------------------- Crossover Operators ---------------------------
    def sbx_crossover(self, parent1, parent2, eta=15):
        """
        Simulated Binary Crossover (SBX).

        :return: Two offspring solutions
        """
        u = np.random.rand(self.n_var)
        beta = np.where(u <= 0.5, (2 * u) ** (1 / (eta + 1)),
                        (1 / (2 * (1 - u))) ** (1 / (eta + 1)))
        child1 = 0.5 * ((1 + beta) * parent1 + (1 - beta) * parent2)
        child2 = 0.5 * ((1 - beta) * parent1 + (1 + beta) * parent2)
        xl, xu = self.evaluator.get_bounds()
        return np.clip(child1, xl, xu), np.clip(child2, xl, xu)

    def one_point_crossover(self, parent1, parent2):
        """
        One-Point Crossover.

        :return: Two offspring solutions
        """
        point = np.random.randint(1, self.n_var)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        xl, xu = self.evaluator.get_bounds()
        return np.clip(child1, xl, xu), np.clip(child2, xl, xu)

    def two_point_crossover(self, parent1, parent2):
        """
        Two-Point Crossover.

        :return: Two offspring solutions
        """
        points = np.sort(np.random.choice(range(1, self.n_var), size=2, replace=False))
        p1, p2 = points
        child1 = parent1.copy()
        child2 = parent2.copy()
        child1[p1:p2] = parent2[p1:p2]
        child2[p1:p2] = parent1[p1:p2]
        xl, xu = self.evaluator.get_bounds()
        return np.clip(child1, xl, xu), np.clip(child2, xl, xu)

    def uniform_crossover(self, parent1, parent2):
        """
        Uniform Crossover.

        :return: Two offspring solutions
        """
        mask = np.random.rand(self.n_var) < 0.5
        child1 = np.where(mask, parent1, parent2)
        child2 = np.where(mask, parent2, parent1)
        xl, xu = self.evaluator.get_bounds()
        return np.clip(child1, xl, xu), np.clip(child2, xl, xu)

    # --------------------------- Mutation Operator ---------------------------
    def polynomial_mutation(self, offspring, eta_m=20):
        """
        Polynomial Mutation.

        :param offspring: Array of offspring solutions
        :return: Array of mutated offspring
        """
        xl, xu = self.evaluator.get_bounds()
        mutated = offspring.copy()
        for i in range(mutated.shape[0]):
            for j in range(self.n_var):
                if np.random.rand() < self.mutation_rate:
                    u = np.random.rand()
                    if u < 0.5:
                        delta = (2 * u) ** (1 / (eta_m + 1)) - 1
                    else:
                        delta = 1 - (2 * (1 - u)) ** (1 / (eta_m + 1))
                    mutated[i, j] += delta * (xu[j] - xl[j])
                    mutated[i, j] = np.clip(mutated[i, j], xl[j], xu[j])
        return mutated

    # --------------------- Non-dominated Sorting ---------------------
    def fast_non_dominated_sorting(self, objectives):
        """
        Fast non-dominated sorting.
        :param objectives: Matrix of objective values
        :return: List of fronts (each front is a list of indices)
        """
        pop_size = objectives.shape[0]
        domination_count = np.zeros(pop_size, dtype=int)
        dominated_solutions = [[] for _ in range(pop_size)]
        fronts = [[]]

        for p in range(pop_size):
            for q in range(pop_size):
                if self.dominates(objectives[p], objectives[q]):
                    dominated_solutions[p].append(q)
                elif self.dominates(objectives[q], objectives[p]):
                    domination_count[p] += 1
            if domination_count[p] == 0:
                fronts[0].append(p)

        i = 0
        while fronts[i]:
            next_front = []
            for p in fronts[i]:
                for q in dominated_solutions[p]:
                    domination_count[q] -= 1
                    if domination_count[q] == 0:
                        next_front.append(q)
            i += 1
            fronts.append(next_front)
        return fronts[:-1]

    def dominates(self, obj1, obj2):
        """
        Check if solution with objectives obj1 dominates solution with objectives obj2.
        """
        return np.all(obj1 <= obj2) and np.any(obj1 < obj2)

    def calculate_crowding_distance(self, front, objectives):
        """
        Calculate the crowding distance for a set of solutions in a front.
        :param front: List of indices in the front
        :param objectives: Matrix of objective values
        :return: Array of crowding distances corresponding to the front
        """
        distance = np.zeros(len(front))
        if len(front) == 0:
            return distance

        num_objectives = objectives.shape[1]
        for m in range(num_objectives):
            sorted_indices = np.argsort(objectives[front, m])
            distance[sorted_indices[0]] = np.inf
            distance[sorted_indices[-1]] = np.inf
            min_val = objectives[front[sorted_indices[0]], m]
            max_val = objectives[front[sorted_indices[-1]], m]
            if max_val - min_val == 0:
                continue
            for i in range(1, len(front) - 1):
                distance[sorted_indices[i]] += (
                    objectives[front[sorted_indices[i + 1]], m] -
                    objectives[front[sorted_indices[i - 1]], m]
                ) / (max_val - min_val)
        return distance

    # --------------------- Tournament Selection ---------------------
    def tournament_selection(self, ranks, crowding_distances):
        """
        Binary tournament selection based on rank and crowding distance.
        :return: Array of selected parent indices.
        """
        selected = []
        for _ in range(self.population_size):
            i, j = np.random.choice(self.population_size, 2, replace=False)
            if ranks[i] < ranks[j]:
                selected.append(i)
            elif ranks[i] > ranks[j]:
                selected.append(j)
            else:
                selected.append(i if crowding_distances[i] > crowding_distances[j] else j)
        return np.array(selected)

    # --------------------- Generate Offspring ---------------------
    def generate_offspring(self, parents):
        """
        Generate offspring by applying an adaptive crossover operator (SBX, One-Point,
        Two-Point, or Uniform) and then polynomial mutation. Also record the operator used.
        
        :return: Offspring array and list of corresponding crossover operators used.
        """
        offspring = []
        offspring_ops = []  # Record which crossover operator was used

        for i in range(0, len(parents), 2):
            parent1 = self.population[parents[i]]
            parent2 = self.population[parents[i + 1]]
            if np.random.rand() < self.crossover_rate:
                op = np.random.choice(self.crossover_ops, p=[self.crossover_probs[op] for op in self.crossover_ops])
                if op == "sbx":
                    child1, child2 = self.sbx_crossover(parent1, parent2)
                elif op == "one_point":
                    child1, child2 = self.one_point_crossover(parent1, parent2)
                elif op == "two_point":
                    child1, child2 = self.two_point_crossover(parent1, parent2)
                elif op == "uniform":
                    child1, child2 = self.uniform_crossover(parent1, parent2)
                else:
                    child1, child2 = parent1.copy(), parent2.copy()
                offspring.append(child1)
                offspring.append(child2)
                offspring_ops.extend([op, op])
            else:
                # If no crossover, simply copy the parents.
                child1, child2 = parent1.copy(), parent2.copy()
                offspring.append(child1)
                offspring.append(child2)
                offspring_ops.extend([None, None])
        offspring = np.array(offspring)
        offspring = self.polynomial_mutation(offspring)
        return offspring, offspring_ops

    # --------------------- Merge and Select Next Generation ---------------------
    def merge_and_select(self, offspring, offspring_objectives, offspring_ops):
        """
        Merge the parent and offspring populations and select the next generation using elitism.
        Also returns the indices of individuals selected from the merged population.
        
        :return: Next generation population, objectives, and selected indices.
        """
        combined_population = np.vstack((self.population, offspring))
        combined_objectives = np.vstack((self.objectives, offspring_objectives))

        fronts = self.fast_non_dominated_sorting(combined_objectives)
        new_population = []
        new_objectives = []
        selected_indices = []
        for front in fronts:
            if len(new_population) + len(front) <= self.population_size:
                for idx in front:
                    new_population.append(combined_population[idx])
                    new_objectives.append(combined_objectives[idx])
                    selected_indices.append(idx)
            else:
                distances = self.calculate_crowding_distance(front, combined_objectives)
                sorted_front = np.array(front)[np.argsort(-distances)]
                remaining_slots = self.population_size - len(new_population)
                for idx in sorted_front[:remaining_slots]:
                    new_population.append(combined_population[idx])
                    new_objectives.append(combined_objectives[idx])
                    selected_indices.append(idx)
                break
        return np.array(new_population), np.array(new_objectives), selected_indices

    # --------------------- Main Run Method ---------------------
    def run(self):
        """
        Run Fast-NSGA-II with adaptive crossover selection and diversity injection.
        In problematic cases (e.g., wfg1, wfg2, wfg8), if no improvement in IGD is seen
        for a fixed number of generations, a fraction of the population is reinitialized.
        
        Also logs the evolution of performance metrics and crossover probabilities.
        """
        logger = NSGA2Logger(self.evaluator)
        best_igd = float('inf')
        no_improvement_counter = 0
        injection_threshold = 15      # Number of generations with no IGD improvement before injecting diversity
        injection_fraction = 0.2        # Fraction of population to reinitialize

        for gen in range(self.max_generations):
            # Fast non-dominated sorting to get fronts and assign ranks
            fronts = self.fast_non_dominated_sorting(self.objectives)
            ranks = np.empty(self.population_size, dtype=int)
            for rank, front in enumerate(fronts):
                for idx in front:
                    ranks[idx] = rank

            # Calculate crowding distances for tournament selection
            crowding = np.zeros(self.population_size)
            for front in fronts:
                distances = self.calculate_crowding_distance(front, self.objectives)
                for i, idx in enumerate(front):
                    crowding[idx] = distances[i]

            # Tournament selection for parents
            selected_parents = self.tournament_selection(ranks, crowding)
            offspring, offspring_ops = self.generate_offspring(selected_parents)
            offspring_objectives = self.evaluate_population(offspring)
            self.population, self.objectives, selected_indices = self.merge_and_select(offspring, offspring_objectives, offspring_ops)

            # Adaptive update of crossover operator probabilities based on rewards.
            total_counts = {op: 0 for op in self.crossover_ops}
            selected_counts = {op: 0 for op in self.crossover_ops}
            for op in offspring_ops:
                if op is not None:
                    total_counts[op] += 1
            for idx in selected_indices:
                # Only consider offspring indices (parents are first half of the combined population)
                if idx >= self.population_size:
                    op = offspring_ops[idx - self.population_size]
                    if op is not None:
                        selected_counts[op] += 1
            for op in self.crossover_ops:
                reward = selected_counts[op] / total_counts[op] if total_counts[op] > 0 else 0
                self.crossover_probs[op] = (1 - self.learning_rate) * self.crossover_probs[op] + self.learning_rate * reward
            total_prob = sum(self.crossover_probs.values())
            for op in self.crossover_ops:
                self.crossover_probs[op] /= total_prob

            # Log current generation information including crossover probabilities.
            logger.log(gen, self.population, self.objectives, self.crossover_probs.copy())

            current_igd = logger.igd_values[-1]

            # Diversity injection for problematic cases
            if self.evaluator.problem_name in ["wfg1", "wfg2", "wfg8"]:
                if current_igd < best_igd:
                    best_igd = current_igd
                    no_improvement_counter = 0
                else:
                    no_improvement_counter += 1

                if no_improvement_counter >= injection_threshold:
                    injection_size = int(self.population_size * injection_fraction)
                    print(f"\n[Diversity Injection] No IGD improvement for {injection_threshold} generations. "
                          f"Injecting {injection_size} new individuals.\n")
                    xl, xu = self.evaluator.get_bounds()
                    injected_population = np.random.uniform(xl, xu, (injection_size, self.n_var))
                    injected_population = np.clip(injected_population, xl, xu)
                    injected_objectives = self.evaluate_population(injected_population)
                    combined_population = np.vstack((self.population, injected_population))
                    combined_objectives = np.vstack((self.objectives, injected_objectives))
                    fronts = self.fast_non_dominated_sorting(combined_objectives)
                    new_population = []
                    new_objectives = []
                    for front in fronts:
                        if len(new_population) + len(front) <= self.population_size:
                            new_population.extend(combined_population[front])
                            new_objectives.extend(combined_objectives[front])
                        else:
                            distances = self.calculate_crowding_distance(front, combined_objectives)
                            sorted_front = np.array(front)[np.argsort(-distances)]
                            remaining_slots = self.population_size - len(new_population)
                            new_population.extend(combined_population[sorted_front[:remaining_slots]])
                            new_objectives.extend(combined_objectives[sorted_front[:remaining_slots]])
                            break
                    self.population = np.array(new_population)
                    self.objectives = np.array(new_objectives)
                    no_improvement_counter = 0

            print(f"Generation {gen + 1}/{self.max_generations} completed.")
            print(f"Updated Crossover Probabilities: {self.crossover_probs}\n")

        # Plot logged results: IGD, HV, and Crossover Probabilities evolution.
        logger.plot_results(self.evaluator.problem_name)
        return self.population, self.objectives

    # --------------------- Compute IGD ---------------------
    def compute_igd(self):
        """
        Compute the Inverted Generational Distance (IGD) metric using the true Pareto front.
        """
        true_pareto = self.evaluator.get_true_pareto()
        igd_metric = IGD(true_pareto)
        return igd_metric(self.objectives)

    # --------------------- Compute Hypervolume ---------------------
    def compute_hv(self):
        """
        Compute the Hypervolume (HV) metric using a fixed reference point.
        """
        true_pareto = self.evaluator.get_true_pareto()
        ideal = self.evaluator.ideal_point()
        nadir = self.evaluator.nadir_point()
        if np.any(np.isnan(nadir)) or np.any(nadir < np.max(true_pareto, axis=0)):
            ref_point = np.max(true_pareto, axis=0) + 0.1
        else:
            ref_point = ideal + 1.1 * (nadir - ideal)
        hv_metric = Hypervolume(ref_point)
        hv_value = hv_metric(self.objectives)
        print(f"Hypervolume: {hv_value}")
        return hv_value


###############################################################################
# Logger to Record Performance Metrics and Crossover Probabilities
###############################################################################
class NSGA2Logger:
    def __init__(self, evaluator):
        """
        Logger to record IGD, Hypervolume, and adaptive crossover probabilities over generations.
        """
        self.evaluator = evaluator
        self.igd_values = []
        self.hv_values = []
        self.generations = []
        self.true_pareto = evaluator.get_true_pareto()

        # For fixed reference point used in HV computation.
        ideal = evaluator.ideal_point()
        nadir = evaluator.nadir_point()
        if np.any(np.isnan(nadir)) or np.any(nadir < np.max(self.true_pareto, axis=0)):
            self.fixed_ref_point = np.max(self.true_pareto, axis=0) + 0.1
        else:
            self.fixed_ref_point = ideal + 1.1 * (nadir - ideal)

        # Log the evolution of adaptive crossover probabilities.
        self.crossover_history = {op: [] for op in ["sbx", "one_point", "two_point", "uniform"]}
        # Optionally, also track the best objective value per generation.
        self.best_objectives = []

    def log(self, generation, population, objectives, current_crossover_probs):
        """
        Log the IGD, Hypervolume, and current crossover operator probabilities for a generation.
        """
        igd_metric = IGD(self.true_pareto)
        igd_value = igd_metric(objectives)
        hv_metric = Hypervolume(self.fixed_ref_point)
        hv_value = hv_metric(objectives)
        self.igd_values.append(igd_value)
        self.hv_values.append(hv_value)
        self.generations.append(generation)

        # Log adaptive crossover probabilities.
        for op in self.crossover_history.keys():
            self.crossover_history[op].append(current_crossover_probs.get(op, 0))

        # Log the best (minimum) value for the first objective (as an example).
        best_obj = np.min(objectives[:, 0])
        self.best_objectives.append(best_obj)

        print(f"Generation {generation}: IGD = {igd_value:.4f}, HV = {hv_value:.4f}")

    def plot_results(self, problem_name):
        """
        Plot the evolution of Hypervolume, IGD, adaptive crossover probabilities, and best objective value.
        Save the performance plot as an image file.
        """
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Plot Hypervolume over generations.
        axes[0, 0].plot(self.generations, self.hv_values, color='cyan', linestyle='-', marker='o', label="Hypervolume")
        axes[0, 0].set_xlabel("Generation")
        axes[0, 0].set_ylabel("Hypervolume")
        axes[0, 0].set_title("Hypervolume over Generations")
        axes[0, 0].legend()
        axes[0, 0].grid(True, color='gray', linestyle='--', alpha=0.5)

        # Plot IGD over generations.
        axes[0, 1].plot(self.generations, self.igd_values, color='magenta', linestyle='-', marker='o', label="IGD")
        axes[0, 1].set_xlabel("Generation")
        axes[0, 1].set_ylabel("IGD")
        axes[0, 1].set_title("IGD over Generations")
        axes[0, 1].legend()
        axes[0, 1].grid(True, color='gray', linestyle='--', alpha=0.5)

        # Plot the evolution of adaptive crossover probabilities.
        for op, color in zip(self.crossover_history.keys(), ['orange', 'lime', 'deepskyblue', 'violet']):
            axes[1, 0].plot(self.generations, self.crossover_history[op],
                            color=color, linestyle='-', marker='o', label=op.upper())
        axes[1, 0].set_xlabel("Generation")
        axes[1, 0].set_ylabel("Crossover Probability")
        axes[1, 0].set_title("Adaptive Crossover Probabilities")
        axes[1, 0].legend()
        axes[1, 0].grid(True, color='gray', linestyle='--', alpha=0.5)

        # Plot best (minimum) value of the first objective over generations.
        axes[1, 1].plot(self.generations, self.best_objectives, color='gold', linestyle='-', marker='o', label="Best Objective 1")
        axes[1, 1].set_xlabel("Generation")
        axes[1, 1].set_ylabel("Best Objective 1")
        axes[1, 1].set_title("Best Objective (Min. of Objective 1) over Generations")
        axes[1, 1].legend()
        axes[1, 1].grid(True, color='gray', linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.savefig(f"{problem_name}_performance.png")
        plt.show()


###############################################################################
# Main function to run experiments on WFG problems
###############################################################################
def main():
    # List of WFG problems (wfg1 to wfg9)
    wfg_problems = [f"wfg{i}" for i in range(1, 10)]

    # Algorithm settings (adjust these for longer runs if desired)
    population_size = 100       # Increase for better diversity
    n_var = 10                  # Number of decision variables
    n_obj = 2                   # Number of objectives
    max_generations = 1000      # Increase for longer runs (e.g., 5000 generations)
    crossover_rate = 0.95
    mutation_rate = 1.0 / n_var # Typically set as 1/n_var for polynomial mutation

    results = []

    # Run the algorithm for each problem
    for problem_name in wfg_problems:
        print(f"\nRunning Fast-NSGA-II on {problem_name}...")
        evaluator = WFGEvaluator(problem_name=problem_name, n_var=n_var, n_obj=n_obj)

        # Attempt to retrieve the true Pareto front.
        try:
            true_pareto = evaluator.get_true_pareto()
        except Exception as e:
            print(f"Error retrieving true Pareto front for {problem_name}: {e}")
            continue

        # Initialize and run Fast-NSGA-II.
        fast_nsga = FastNSGAII(
            population_size=population_size,
            n_var=n_var,
            n_obj=n_obj,
            max_generations=max_generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            evaluator=evaluator
        )

        final_population, final_objectives = fast_nsga.run()

        # Compute performance metrics.
        try:
            igd_score = fast_nsga.compute_igd()
            hv_score = fast_nsga.compute_hv()
        except Exception as e:
            print(f"Error computing metrics for {problem_name}: {e}")
            continue

        results.append({
            "Problem": problem_name,
            "IGD": igd_score,
            "HV": hv_score,
            "True Pareto": true_pareto,
            "Obtained Pareto": final_objectives
        })

        print(f"{problem_name} - IGD: {igd_score:.4f}, HV: {hv_score:.4f}")

        # Plot Pareto front comparison.
        plt.figure(figsize=(8, 6))
        plt.scatter(true_pareto[:, 0], true_pareto[:, 1], color='deepskyblue',
                    label="True Pareto Front", alpha=0.6, s=60)
        plt.scatter(final_objectives[:, 0], final_objectives[:, 1], color='red',
                    label="Obtained Solutions", alpha=0.6, s=60)
        plt.xlabel("Objective 1")
        plt.ylabel("Objective 2")
        plt.title(f"Pareto Front Comparison for {problem_name}")
        plt.legend()
        plt.grid(True, color='gray', linestyle='--', alpha=0.5)
        plt.show()

    # Print summary of results.
    print("\nSummary of Results:")
    for res in results:
        print(f"{res['Problem']}: IGD = {res['IGD']:.4f}, HV = {res['HV']:.4f}")


# Run the main function
if __name__ == "__main__":
    main()
